# MEME FIMO Motif Scan

`run_meme_fimo_scan` uses MEME Suite [FIMO](https://meme-suite.org/meme/doc/fimo.html) (Find Individual Motif Occurrences) to scan sequences for occurrences of known motifs (position weight matrices) supplied in a [MEME](https://meme-suite.org/meme/) `.meme` file. It is backed by [`pymemesuite`](https://pypi.org/project/pymemesuite/), so no host MEME installation is required.

FIMO is described in Grant, Bailey & Noble (2011), ["FIMO: scanning for occurrences of a given motif"](https://doi.org/10.1093/bioinformatics/btr064), *Bioinformatics* 27(7):1017-1018.

In [1]:
from pathlib import Path

import proto_tools
from proto_tools.tools.gene_annotation.meme import (
    MEMEFimoScanInput,
    MEMEFimoScanConfig,
    run_meme_fimo_scan,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Display input docs
display_api_reference("meme-fimo-scan", "input", "run_meme_fimo_scan")

# Bundled MEME motif: the S. pyogenes SF370 CRISPR1 direct repeat (derived from the 7
# repeat instances in the locus below).
_meme_dir = Path(proto_tools.__file__).resolve().parent / "tools" / "gene_annotation" / "meme" / "examples"
motif_path = _meme_dir / "example.meme"

# Scan the real CRISPR1 locus for occurrences of the repeat motif.
dna_seq = "".join(l for l in (_meme_dir / "spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))

inputs = MEMEFimoScanInput(
    sequences=dna_seq,
    motifs=str(motif_path),
)

**Input** — `MEMEFimoScanInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Target sequences to scan, as a single string or a list of strings |
| <code>motifs</code> | <code>str &#124; Path</code> | required | Path to a MEME-format (.meme) motif PWM file, e.g. from JASPAR. AssetRef supported |

In [3]:
# Display config docs
display_api_reference("meme-fimo-scan", "config", "run_meme_fimo_scan")

# Default config: p-value threshold 1e-4, scanning both strands | see docs above for all fields
config = MEMEFimoScanConfig()

**Config** — `MEMEFimoScanConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>threshold</code> | <code>float</code> | <code>0.0001</code> | Report matches with p-value <= this cutoff; lower is stricter (FIMO --thresh) |
| <code>both_strands</code> | <code>bool</code> | <code>True</code> | Scan both strands for nucleotide motifs; set False for single-strand. Ignored for protein |

In [4]:
# Display output docs
display_api_reference("meme-fimo-scan", "output", "run_meme_fimo_scan")

**Output** — `MEMEFimoScanOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[FimoSequenceMatches]</code> | <code>[]</code> | Per-sequence motif occurrences, aligned to the input sequences |

**`FimoSequenceMatches`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>matches</code> | <code>list[FimoMatch]</code> | <code>[]</code> | Motif occurrences found in this sequence |

**`FimoMatch`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>motif_id</code> | <code>str</code> | required | Matched motif identifier (motif accession when present) |
| <code>motif_alt_id</code> | <code>str</code> | required | Alternate motif name; '-' if none |
| <code>start</code> | <code>int</code> | required | Match start in the target sequence (1-indexed, inclusive) |
| <code>stop</code> | <code>int</code> | required | Match end in the target sequence (1-indexed, inclusive) |
| <code>strand</code> | <code>str</code> | required | Strand of the match ('+' or '-') |
| <code>score</code> | <code>float</code> | required | Log-odds score of the match |
| <code>pvalue</code> | <code>float</code> | required | P-value of the match |
| <code>qvalue</code> | <code>float</code> | required | Benjamini-Hochberg q-value (FDR) of the match |
| <code>matched_sequence</code> | <code>str</code> | required | Subsequence at the hit on the matched strand |

## Basic usage

Scan the DNA sequence against the example motif with the default configuration. The two consensus copies should each produce a match.

In [5]:
result = run_meme_fimo_scan(inputs, config)

print(f"Found {result.num_matches} motif occurrence(s)")

for match in result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}"
    )

Running run_meme_fimo_scan [00:00]

Found 9 motif occurrence(s)
  motif=SPYO_CRISPR1, start=4819, stop=4854, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=4885, stop=4920, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=4951, stop=4986, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=5017, stop=5052, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=5083, stop=5118, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=5149, stop=5184, strand=+, pvalue=3.60e-22
  motif=SPYO_CRISPR1, start=5215, stop=5250, strand=+, pvalue=2.44e-18
  motif=SPYO_CRISPR1, start=5165, stop=5200, strand=-, pvalue=5.49e-05
  motif=SPYO_CRISPR1, start=206, stop=241, strand=+, pvalue=6.70e-05


## Advanced usage

Set `both_strands=False` to scan only the forward strand (FIMO `--norc`), and loosen `threshold` to report weaker matches.

In [6]:
advanced_config = MEMEFimoScanConfig(
    threshold=1e-3,
    both_strands=False,
)

advanced_result = run_meme_fimo_scan(inputs, advanced_config)

print(f"Found {advanced_result.num_matches} motif occurrence(s) on the forward strand")

for match in advanced_result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}, matched={match.matched_sequence}"
    )

Running run_meme_fimo_scan [00:00]

Found 16 motif occurrence(s) on the forward strand
  motif=SPYO_CRISPR1, start=4819, stop=4854, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=4885, stop=4920, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=4951, stop=4986, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=5017, stop=5052, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=5083, stop=5118, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=5149, stop=5184, strand=+, pvalue=3.60e-22, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC
  motif=SPYO_CRISPR1, start=5215, stop=5250, strand=+, pvalue=2.44e-18, matched=GTTTTAGAGCTATGCTGTTTTGAATGGTCTCCATTC
  motif=SPYO_CRISPR1, start=206, stop=241, strand=+, pvalue=6.70e-05, matched=GCTTCAGCTCAATCATTTATTGAACGCATGACAAAC
  motif=SPYO_CR

## Export results

FIMO matches can be exported to CSV (one row per match) or JSON. CSV is the default format.

In [7]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="meme_fimo_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
